# Step-by-step Bayesian regularization tutorial

This notebook makes every stage of the two-dimensional dijet regularization study visible. Response-training MC supplies the migration matrix, generator and reconstructed marginals, purity, efficiency, and Bayesian prior. An independent toy-truth model is normalized once to the MinimumBias exposure, forward-folded, assigned the relative statistical uncertainties measured in data, fluctuated into one common seeded toy ensemble, and unfolded.

Every eta-space comparison is written separately for every configured $p_T^{ave}$ interval. `SELECTED_TOY_NUMBERS` uses human-facing one-based labels (default toys 1, 3, and 5), and those same pseudo-experiments are retained in all downstream plots. `DISPLAY_ITERATIONS` chooses the iteration curves shown before the MSE optimization; the scan itself still evaluates every iteration through `MAX_ITERATIONS`.

The workflow uses the factorized correction convention

$$m^{all}\xrightarrow{\times P}m^{matched}\xrightarrow{U_n[M]}\hat t^{matched}_n\xrightarrow{/\epsilon}\hat t^{all}_n,$$

with fixed response, purity, and efficiency. Only the copied experimental statistical uncertainty is sampled by the toy ensemble.


Input resolution uses reusable files for the requested jet-selection stem, then the original ROOT file, and raises an error if neither exists. Source eta-cut tables must match the current producer; stale inputs require regeneration. `ETA_CUT_VALUE` (where present) is the physical per-jet acceptance, default 1.9; its index is derived from the shared table. `SINGLE_JET_BINNING` and `DIJET_BINNING` independently select existing `normal` or `test` intervals. Existing `PTAVE_BIN_SET` controls use `standard` as the name for normal binning. Trigger-specific normal bins remain the notebook defaults; test dijet bins use MB 50–100, Jet80 100–180, and Jet100 above 180 GeV. Jet60 has no test interval. Diagnostic frame scans and trigger-stitching overlap windows retain their established definitions.

## Environment and reusable plotting helpers


In [ ]:
# Cell role: initialize ROOT, RooUnfold, project helpers, and CMS plotting defaults.
# Interpretation: no physics histogram is modified in this cell.
%load_ext autoreload
%autoreload 2

from pathlib import Path
import math
import os
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'CMakeLists.txt').is_file()
                     and (p / 'hist_analysis').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Start Jupyter from the jetAnalysis repository root')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root, load_roounfold
ROOT = load_root(batch=True)
load_roounfold(ROOT, project_root=PROJECT_ROOT)
import RooUnfold
ROOT.TH1.AddDirectory(False)
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import DIJET_PTAVE_BINS, TEST_DIJET_PTAVE_BINS, STANDARD_DIJET_ETA_CUT_INDEX
from hist_analysis.python.data_trigger_stitching import load_trigger_stitching_config
from hist_analysis.python.histogram_io import (
    load_histogram, resolve_combined_file, resolve_data_file, resolve_direction_file,
    normalize_unfolding_direction, resolve_unfolding_mc_file, resolve_unfolding_truth_file,
)
from hist_analysis.python.plot_labels import CMS_SIMULATION_HEADER, PPB_HEADER_RIGHT
from hist_analysis.python.plotting import draw_closure, draw_overlay
from hist_analysis.python.root_style import (
    configure_cms_header, configure_mplhep_export, draw_cms_header, save_canvas,
    set_2d_style, set_pad_style,
)
from hist_analysis.python.unfolding import (
    UnfoldingInputKeys, apply_purity_correction, as_pt_intervals,
    build_roounfold_response, calculate_response_diagnostics,
    copy_histogram_relative_errors, extract_eta_block,
    flatten_pt_eta_projections, flatten_sparse_response,
    flattened_pt_range_integral, forward_fold_truth, load_unfolding_inputs,
    make_gaussian_toys, prepare_factorized_corrections, project_eta_by_pt,
    scan_bayes_regularization, unfold_bayes, validate_response_accounting,
)
from hist_analysis.python.unfolding_plots import draw_regularization_metrics

CMS_HEADER_LEFT = CMS_SIMULATION_HEADER
CMS_HEADER_RIGHT = PPB_HEADER_RIGHT
SAVE_MPLHEP = False
SAVE_MPLHEP_PNG = False
configure_cms_header(CMS_HEADER_LEFT, CMS_HEADER_RIGHT)
configure_mplhep_export(SAVE_MPLHEP, save_png=SAVE_MPLHEP_PNG)


def pt_tag(interval):
    return f'ptave_{interval[0]:g}_{interval[1]:g}'.replace('.', 'p')


def pt_title(interval):
    return f'{interval[0]:g} < p_{{T}}^{{ave}} < {interval[1]:g} GeV'


def detached_clone(histogram, name):
    clone = histogram.Clone(name)
    clone.SetDirectory(0)
    return clone


def unit_normalized(histogram, name):
    clone = detached_clone(histogram, name)
    integral = clone.Integral()
    if not math.isfinite(integral) or integral <= 0.0:
        raise ValueError(f'Cannot unit-normalize {histogram.GetName()}: integral={integral:g}')
    clone.Scale(1.0 / integral)
    return clone


def eta_block(histogram, template, pt_index, name):
    return extract_eta_block(histogram, template, layout, pt_index, name=name)


def draw_per_pt(distributions, *, output_stem, y_title, ratio_nominal=None,
                ratio_range=(0.5, 1.5), log_y=False, curve_styles=None):
    canvases = []
    objects = []
    for pt_index, interval in enumerate(PTAVE_INTERVALS):
        curves = distributions(pt_index)
        for histogram in curves.values():
            histogram.GetYaxis().SetNoExponent(True)
        output = OUTPUT_DIR / f'{output_stem}_{pt_tag(interval)}.pdf'
        canvas_name = f'canvas_{output_stem}_{pt_index}'
        if ratio_nominal is None:
            canvas = draw_overlay(
                curves, title='', x_title='#eta_{CM}',
                y_title=y_title, log_y=log_y, output=output,
                annotations=(pt_title(interval),), canvas_name=canvas_name,
                curve_styles=curve_styles)
            derived = None
        else:
            canvas, derived = draw_closure(
                curves, nominal=ratio_nominal, title='',
                x_title='#eta_{CM}', y_title=y_title,
                ratio_range=ratio_range, ratio_y_title='Ratio to nominal',
                draw_nominal_ratio=False, log_y=log_y,
                annotations=(pt_title(interval),),
                output=output, canvas_name=canvas_name,
                curve_styles=curve_styles)
        canvases.append(canvas)
        objects.append((curves, derived))
        display(canvas)
    return canvases, objects


## Configuration


In [ ]:
ETA_CUT_VALUE = 1.9  # per-jet |etaCM| <= value; supported producer cuts only
from hist_analysis.config.histograms import pt_bins, eta_cut_index
# Cell role: define user-facing inputs, response choice, toys, and displayed iterations.
# Interpretation: changing these values changes the statistical study.
RESPONSE_DATASET = 'pythia_cp5'  # embedding, pythia, or pythia_cp5
RESPONSE_DIRECTION = os.environ.get('UNFOLD2D_RESPONSE_DIRECTION', 'combined')
STITCHING_DIRECTION = 'combined'
RESPONSE_FILE_STEM = 'jetId'
RESPONSE_VARIANT = 'nominal'
RESPONSE_KEY_TEMPLATES = {
    'jer_def_extra': {
        'truth': 'hGenDijetPtEtaCM_{index}',
        'measured': 'hRecoDijetPtEtaCMJerDefExtraUnfold_{index}',
        'response': 'hGenDijetPtEtaCMVsRecoJerDefExtraPtEtaCM_{index}',
        'miss': 'hGenDijetPtEtaCMMissJerDefExtra_{index}',
        'fake': 'hRecoDijetPtEtaCMFakeJerDefExtra_{index}',
    },
    'nominal': {
        'truth': 'hGenDijetPtEtaCM_{index}',
        'measured': 'hRecoDijetPtEtaCMJerDefUnfold_{index}',
        'response': 'hGenDijetPtEtaCMVsRecoPtEtaCM_{index}',
        'miss': 'hGenDijetPtEtaCMMiss_{index}', 'fake': 'hRecoDijetPtEtaCMFake_{index}',
    },
    'controlled': {
        'truth': 'hGenDijetPtEtaCMControlledSample_{index}',
        'measured': 'hRecoDijetPtEtaCMJerDefUnfoldControlledSample_{index}',
        'response': 'hGenDijetPtEtaCMVsRecoPtEtaCMControlledSample_{index}',
        'miss': 'hGenDijetPtEtaCMMissControlledSample_{index}',
        'fake': 'hRecoDijetPtEtaCMFakeControlledSample_{index}',
    },
    'studied': {
        'truth': 'hGenDijetPtEtaCMStudiedSample_{index}',
        'measured': 'hRecoDijetPtEtaCMJerDefUnfoldStudiedSample_{index}',
        'response': 'hGenDijetPtEtaCMVsRecoPtEtaCMStudiedSample_{index}',
        'miss': 'hGenDijetPtEtaCMMissStudiedSample_{index}',
        'fake': 'hRecoDijetPtEtaCMFakeStudiedSample_{index}',
    },
}
DATA_SELECTION = 'jetId'
TOY_TRUTH_MODEL = 'pythia_cuep8m1' # embedding, pythia, pythia_cuep8m1, pythia_cp5, pythia_cp5_standalone, response_gen, mixed_prior, or custom
# Direction is used by Embedding/PYTHIA and ignored for pp tune samples.
TOY_TRUTH_DIRECTION = 'pp'
# Optional overrides make any compatible TH2 truth distribution usable.
# Set TOY_TRUTH_MODEL='custom' when using a file that is not one of the named samples.
TOY_TRUTH_FILE = os.environ.get('UNFOLD2D_TOY_TRUTH_FILE')
TOY_TRUTH_HISTOGRAM = os.environ.get('UNFOLD2D_TOY_TRUTH_HISTOGRAM')
ETA_CUT_INDEX = eta_cut_index(ETA_CUT_VALUE)
ETA_CUT = ETA_CUT_VALUE
ETA_RANGE = (-ETA_CUT, ETA_CUT)
PTAVE_BIN_SET = 'test'  # test or standard
PTAVE_BIN_SETS = {'test': TEST_DIJET_PTAVE_BINS, 'standard': DIJET_PTAVE_BINS}
if PTAVE_BIN_SET not in PTAVE_BIN_SETS:
    raise ValueError(f'Unsupported PTAVE_BIN_SET={PTAVE_BIN_SET!r}')
PTAVE_INTERVALS = as_pt_intervals(PTAVE_BIN_SETS[PTAVE_BIN_SET])

N_TOYS = 50
MAX_ITERATIONS = 6
RANDOM_SEED = 12345
SELECTED_TOY_NUMBERS = (1, 3, 5)  # one-based labels used in plots
DISPLAY_ITERATIONS = (1, 3, 5)
RESPONSE_SCALE = 1.0e12
RESPONSE_COMPONENT_TOLERANCE = 1.0e-5
RESPONSE_ACCOUNTING_TOLERANCE = 1.0e-9
RESAMPLE_NEGATIVE_TOY_BIN = True

STANDARD_TRIGGER_BY_INTERVAL = {
    (50.0, 60.0): 'MinimumBias', (60.0, 70.0): 'MinimumBias',
    (70.0, 80.0): 'MinimumBias', (80.0, 90.0): 'Jet60',
    (90.0, 100.0): 'Jet60', (100.0, 110.0): 'Jet80',
    (110.0, 120.0): 'Jet80', (120.0, 130.0): 'Jet100',
    (130.0, 140.0): 'Jet100', (140.0, 150.0): 'Jet100',
    (150.0, 160.0): 'Jet100', (160.0, 180.0): 'Jet100',
    (180.0, 200.0): 'Jet100', (200.0, 250.0): 'Jet100',
    (250.0, 300.0): 'Jet100', (300.0, 500.0): 'Jet100',
    (500.0, 1000.0): 'Jet100',
}
from hist_analysis.config.histograms import TEST_DIJET_TRIGGER_BY_INTERVAL
TEST_TRIGGER_BY_INTERVAL = dict(TEST_DIJET_TRIGGER_BY_INTERVAL)
TRIGGER_BY_INTERVAL = (TEST_TRIGGER_BY_INTERVAL if PTAVE_BIN_SET == 'test'
                       else STANDARD_TRIGGER_BY_INTERVAL)

if TOY_TRUTH_MODEL not in {
        'response_gen', 'mixed_prior', 'embedding', 'pythia',
        'pythia_cuep8m1', 'pythia_cp5', 'pythia_cp5_standalone', 'custom'}:
    raise ValueError(f'Unsupported TOY_TRUTH_MODEL={TOY_TRUTH_MODEL!r}')
if TOY_TRUTH_MODEL == 'custom' and not TOY_TRUTH_FILE:
    raise ValueError("TOY_TRUTH_MODEL='custom' requires UNFOLD2D_TOY_TRUTH_FILE")
if RESPONSE_DATASET not in {'embedding', 'pythia', 'pythia_cp5'}:
    raise ValueError("RESPONSE_DATASET must be embedding, pythia, or pythia_cp5")
RESPONSE_DIRECTION = normalize_unfolding_direction(
    RESPONSE_DATASET, RESPONSE_DIRECTION)
if TOY_TRUTH_MODEL in {'response_gen', 'mixed_prior'}:
    TOY_TRUTH_DIRECTION = RESPONSE_DIRECTION
if TOY_TRUTH_MODEL in {
        'embedding', 'pythia', 'pythia_cuep8m1', 'pythia_cp5',
        'pythia_cp5_standalone'}:
    TOY_TRUTH_DIRECTION = normalize_unfolding_direction(
        TOY_TRUTH_MODEL, TOY_TRUTH_DIRECTION)
if RESPONSE_DIRECTION not in {'combined', 'Pbgoing', 'pgoing', 'pp'}:
    raise ValueError("RESPONSE_DIRECTION must be 'combined', 'Pbgoing', or 'pgoing'")
if RESPONSE_VARIANT not in RESPONSE_KEY_TEMPLATES:
    raise ValueError(f'Unsupported RESPONSE_VARIANT={RESPONSE_VARIANT!r}')
if len(set(SELECTED_TOY_NUMBERS)) != len(SELECTED_TOY_NUMBERS):
    raise ValueError('SELECTED_TOY_NUMBERS contains duplicates')
if any(number < 1 or number > N_TOYS for number in SELECTED_TOY_NUMBERS):
    raise IndexError(f'SELECTED_TOY_NUMBERS must lie in [1, {N_TOYS}]')
if any(number < 1 or number > MAX_ITERATIONS for number in DISPLAY_ITERATIONS):
    raise IndexError(f'DISPLAY_ITERATIONS must lie in [1, {MAX_ITERATIONS}]')
SELECTED_TOY_INDICES = tuple(number - 1 for number in SELECTED_TOY_NUMBERS)

STITCHING_DIR = (PROJECT_ROOT / 'hist_analysis/output/data_trigger_stitching'
                 / STITCHING_DIRECTION / DATA_SELECTION / 'dijet_cm')
candidates = sorted(STITCHING_DIR.glob('trigger_stitching_weights_*.json'))
compatible = []
stitching_rejections = []
for candidate in candidates:
    try:
        compatible.append(load_trigger_stitching_config(
            candidate, direction=STITCHING_DIRECTION, selection=DATA_SELECTION,
            observable='dijet', frame='cm', eta_range=ETA_RANGE,
            eta_cut_index=ETA_CUT_INDEX))
    except (FileNotFoundError, ValueError) as error:
        stitching_rejections.append(str(error))
if len(compatible) != 1:
    details = '\n'.join(f'  {message}' for message in stitching_rejections)
    raise RuntimeError(
        f'Expected one compatible stitching artifact in {STITCHING_DIR}, '
        f'found {len(compatible)}.' + (f'\nRejected candidates:\n{details}' if details else ''))
STITCHING_CONFIG = compatible[0]
mb_records = [r for r in STITCHING_CONFIG.metadata['scales']
              if r.get('reference') in {'MB', 'MinimumBias'}]
if len(mb_records) != 1:
    raise RuntimeError('Expected one direct MB stitching record')
MB_NORMALIZATION_RANGE = tuple(float(x) for x in mb_records[0]['pt_range'])
NORMALIZATION_RANGE = (
    MB_NORMALIZATION_RANGE if PTAVE_BIN_SET == 'standard'
    else next(interval for interval in PTAVE_INTERVALS
              if TRIGGER_BY_INTERVAL[interval] == 'MinimumBias'))
OUTPUT_DIR = (PROJECT_ROOT / 'hist_analysis/output/unfold2D_regularization_tutorial'
              / RESPONSE_DATASET / RESPONSE_DIRECTION / RESPONSE_VARIANT
              / f'truth_{TOY_TRUTH_MODEL}_{TOY_TRUTH_DIRECTION}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Selected one-based toy numbers: {SELECTED_TOY_NUMBERS}')
print(f'Displayed Bayesian iterations: {DISPLAY_ITERATIONS}')


## 1. Response model, generator/reconstructed marginals, purity, and efficiency

The flattened response matrix is shown first. Then each $p_T^{ave}$ interval gets a response-training Gen/Reco overlay and a purity/efficiency overlay. The latter are fixed corrections: purity is matched Reco divided by inclusive Reco, while efficiency is matched Gen divided by inclusive Gen.


In [ ]:
# Cell role: construct the response and plot every response diagnostic requested in step 1.
response_file = resolve_unfolding_mc_file(
    BASE_DIR, RESPONSE_DATASET, RESPONSE_DIRECTION, RESPONSE_FILE_STEM)
templates = RESPONSE_KEY_TEMPLATES[RESPONSE_VARIANT]
keys = UnfoldingInputKeys(**{
    role: template.format(index=ETA_CUT_INDEX) for role, template in templates.items()})
inputs = load_unfolding_inputs(response_file, keys)
if TOY_TRUTH_FILE:
    toy_truth_file = Path(TOY_TRUTH_FILE).expanduser()
elif TOY_TRUTH_MODEL in {'response_gen', 'mixed_prior'}:
    toy_truth_file = response_file
else:
    toy_truth_file = resolve_unfolding_truth_file(
        BASE_DIR, TOY_TRUTH_MODEL, TOY_TRUTH_DIRECTION, RESPONSE_FILE_STEM)
default_truth_key = (
    f'hGenDijetPtEtaCMMixedPrior_{ETA_CUT_INDEX}'
    if TOY_TRUTH_MODEL == 'mixed_prior'
    else f'hGenDijetPtEtaCM_{ETA_CUT_INDEX}')
toy_truth_key = TOY_TRUTH_HISTOGRAM or default_truth_key
toy_truth_2d = load_histogram(str(toy_truth_file), toy_truth_key)
print(f'Toy truth: {TOY_TRUTH_MODEL} from {toy_truth_file}:{toy_truth_key}')

project = lambda h, prefix: project_eta_by_pt(
    h, PTAVE_INTERVALS, name_prefix=prefix, eta_range=ETA_RANGE)
training_truth_by_pt = project(inputs.truth, 'hTutorialTrainingTruth')
training_reco_by_pt = project(inputs.measured, 'hTutorialTrainingReco')
miss_by_pt = project(inputs.miss, 'hTutorialMiss')
fake_by_pt = project(inputs.fake, 'hTutorialFake')
toy_truth_by_pt = project(toy_truth_2d, 'hTutorialToyTruth')
training_truth, layout = flatten_pt_eta_projections(
    training_truth_by_pt, name='hTutorialTrainingTruthFlat', pt_bins=PTAVE_INTERVALS)
training_reco, _ = flatten_pt_eta_projections(
    training_reco_by_pt, name='hTutorialTrainingRecoFlat', layout=layout)
training_miss, _ = flatten_pt_eta_projections(
    miss_by_pt, name='hTutorialMissFlat', layout=layout)
training_fake, _ = flatten_pt_eta_projections(
    fake_by_pt, name='hTutorialFakeFlat', layout=layout)
toy_truth_unscaled, _ = flatten_pt_eta_projections(
    toy_truth_by_pt, name='hTutorialToyTruthFlat', layout=layout)
response_matrix, _ = flatten_sparse_response(
    inputs.response, PTAVE_INTERVALS, name='hTutorialResponse',
    layout=layout, eta_range=ETA_RANGE)
diagnostics = calculate_response_diagnostics(
    response_matrix, training_truth, training_reco,
    explicit_miss=training_miss, explicit_fake=training_fake,
    tolerance=RESPONSE_COMPONENT_TOLERANCE)
response_accounting = validate_response_accounting(
    response_matrix, training_truth, training_reco, diagnostics,
    tolerance=RESPONSE_ACCOUNTING_TOLERANCE)
inclusive_response = build_roounfold_response(
    RooUnfold, training_truth, training_reco, response_matrix,
    diagnostics=diagnostics, scale=RESPONSE_SCALE, require_fakes=True,
    validation_tolerance=RESPONSE_COMPONENT_TOLERANCE,
    name='tutorialInclusiveResponse')
corrections = prepare_factorized_corrections(
    training_reco, training_reco, diagnostics.matched_measured,
    training_truth, diagnostics.matched_truth, name_prefix='hTutorial',
    tolerance=RESPONSE_COMPONENT_TOLERANCE)
matched_response = build_roounfold_response(
    RooUnfold, diagnostics.matched_truth, diagnostics.matched_measured,
    response_matrix, scale=RESPONSE_SCALE, require_fakes=False,
    name='tutorialMatchedResponse')

response_canvas = ROOT.TCanvas('canvas_tutorial_response_matrix', '', 900, 800)
set_pad_style(response_canvas, grid_x=False, grid_y=False)
response_canvas.SetRightMargin(0.16)
set_2d_style(response_matrix)
response_matrix.SetTitle('Flattened detector response;Reco global bin;Truth global bin')
response_matrix.Draw('COLZ')
response_header = draw_cms_header(
    response_canvas, CMS_HEADER_LEFT, CMS_HEADER_RIGHT)
response_canvas.Update()
save_canvas(response_canvas, OUTPUT_DIR / '01_response_matrix.pdf')
response_canvas._objects = [response_matrix, *response_header]
display(response_canvas)

step1_marginal_canvases, step1_marginal_objects = draw_per_pt(
    lambda i: {
        'Response Gen': training_truth_by_pt[i],
        'Response Reco': training_reco_by_pt[i],
    }, output_stem='01_response_gen_reco', y_title='Weighted entries',
    ratio_nominal='Response Gen', ratio_range=(0.5, 1.5), log_y=False,
    curve_styles={'Response Gen': (ROOT.kBlue, 21),
                  'Response Reco': (ROOT.kRed, 20)})

step1_correction_canvases, step1_correction_objects = draw_per_pt(
    lambda i: {
        'Purity': eta_block(corrections.purity, training_reco_by_pt[i], i,
                            f'hTutorialPurityEta{i}'),
        'Efficiency': eta_block(corrections.efficiency, training_truth_by_pt[i], i,
                                f'hTutorialEfficiencyEta{i}'),
    }, output_stem='01_purity_efficiency', y_title='Correction factor',
    ratio_nominal='Purity', ratio_range=(0.5, 1.5))

print(f'Response sparsity: {response_accounting.response_sparsity:.3%}')
print(f'Integrated truth efficiency: {response_accounting.global_efficiency:.6g}')
print(f'Integrated reco fake fraction: {response_accounting.global_fake_fraction:.6g}')


## 2. Self-normalized toy truth before forward folding

Each unscaled toy-truth eta projection is divided by its own integral. This is a shape-only view; the flattened histogram used later remains unchanged.


In [ ]:
# Cell role: plot the pre-folding toy truth with unit integral in every pTave interval.
toy_truth_unit_by_pt = [
    unit_normalized(histogram, f'hTutorialToyTruthUnit{i}')
    for i, histogram in enumerate(toy_truth_by_pt)
]
step2_canvases, step2_objects = draw_per_pt(
    lambda i: {'Toy truth': toy_truth_unit_by_pt[i]},
    output_stem='02_toy_truth_self_normalized',
    y_title='1/N dN/d#eta_{CM}')


## 3. Self-normalized toy truth compared with response-training Gen

Both curves are normalized independently within each $p_T^{ave}$ interval, so this plot isolates the prior-shape difference that Bayesian regularization must overcome.


In [ ]:
# Cell role: compare the unit-area target truth with the response-training Gen prior.
training_truth_unit_by_pt = [
    unit_normalized(histogram, f'hTutorialTrainingTruthUnit{i}')
    for i, histogram in enumerate(training_truth_by_pt)
]
step3_canvases, step3_objects = draw_per_pt(
    lambda i: {
        'Response-training Gen': training_truth_unit_by_pt[i],
        'Toy truth': toy_truth_unit_by_pt[i],
    }, output_stem='03_toy_truth_vs_gen_self_normalized',
    y_title='1/N dN/d#eta_{CM}', ratio_nominal='Response-training Gen',
    curve_styles={'Response-training Gen': (ROOT.kBlue, 21),
                  'Toy truth': (ROOT.kRed, 20)})


## 4. Globally rescaled truth and its forward-folded partner

The unscaled toy truth is folded once. A single scale factor from the direct MinimumBias overlap then rescales the complete truth distribution, preserving all relative $p_T^{ave}$ and eta shapes. The rescaled truth and its detector-level partner are overlaid in each interval.


In [ ]:
# Cell role: stitch data, determine one global exposure scale, and forward-fold the scaled truth.
data_maps = {}
data_by_pt = []
for pt_index, interval in enumerate(PTAVE_INTERVALS):
    trigger = TRIGGER_BY_INTERVAL[interval]
    if trigger not in data_maps:
        filename = resolve_data_file(BASE_DIR / 'exp', trigger, STITCHING_DIRECTION, DATA_SELECTION)
        histogram = load_histogram(str(filename), STITCHING_CONFIG.histogram_keys[trigger])
        histogram.Scale(STITCHING_CONFIG.weights[trigger])
        data_maps[trigger] = histogram
    data_by_pt.append(project_eta_by_pt(
        data_maps[trigger], (interval,), name_prefix=f'hTutorialData{pt_index}',
        eta_range=ETA_RANGE)[0])
data_reco, _ = flatten_pt_eta_projections(
    data_by_pt, name='hTutorialStitchedData', layout=layout)

unscaled_fold = forward_fold_truth(
    inclusive_response, toy_truth_unscaled, diagnostics=diagnostics,
    add_fakes=True, fake_normalization='matched_fraction',
    name='hTutorialUnscaledFold')
mb_projection = project_eta_by_pt(
    data_maps['MinimumBias'], (NORMALIZATION_RANGE,),
    name_prefix='hTutorialMBNormalization', eta_range=ETA_RANGE)[0]
mb_yield = mb_projection.Integral()
folded_yield = flattened_pt_range_integral(
    unscaled_fold.histogram, layout, NORMALIZATION_RANGE)
mb_scale = mb_yield / folded_yield
toy_truth = detached_clone(toy_truth_unscaled, 'hTutorialScaledToyTruth')
toy_truth.Scale(mb_scale)
target_fold = forward_fold_truth(
    inclusive_response, toy_truth, diagnostics=diagnostics,
    add_fakes=True, fake_normalization='matched_fraction',
    name='hTutorialForwardFoldedTruth')

scaled_truth_by_pt = [
    eta_block(toy_truth, training_truth_by_pt[i], i, f'hTutorialScaledTruthEta{i}')
    for i in range(len(PTAVE_INTERVALS))]
folded_by_pt = [
    eta_block(target_fold.histogram, training_reco_by_pt[i], i,
              f'hTutorialForwardFoldedEta{i}')
    for i in range(len(PTAVE_INTERVALS))]
step4_canvases, step4_objects = draw_per_pt(
    lambda i: {
        'Rescaled toy truth': scaled_truth_by_pt[i],
        'Forward-folded partner': folded_by_pt[i],
    }, output_stem='04_scaled_truth_and_forward_folded',
    y_title='Entries', ratio_nominal='Rescaled toy truth',
    ratio_range=(0.5, 1.5), log_y=False)
print(f'MB data / unscaled folded toy = {mb_yield:g} / {folded_yield:g}')
print(f'Normalization pTave range = {NORMALIZATION_RANGE}')
print(f'One global truth scale factor = {mb_scale:.10g}')


## 5. Forward-folded distribution before and after copying data uncertainties

The central values must be identical. Only the vertical statistical errors change: the second curve receives the relative bin-by-bin uncertainty of the stitched experimental data. The ratio panel is therefore a central-value identity check, while the error bars show the statistical change.


In [ ]:
# Cell role: transfer experimental relative errors without changing the forward-folded mean.
toy_expectation = copy_histogram_relative_errors(
    target_fold.histogram, data_reco, name='hTutorialToyExpectation',
    source_zero_relative_tolerance=1.0e-14, layout=layout,
    use_nearest_reference_for_empty=True)
expectation_by_pt = [
    eta_block(toy_expectation, training_reco_by_pt[i], i,
              f'hTutorialExpectationEta{i}')
    for i in range(len(PTAVE_INTERVALS))]
step5_canvases, step5_objects = draw_per_pt(
    lambda i: {
        'Response statistical errors': folded_by_pt[i],
        'Data statistical errors': expectation_by_pt[i],
    }, output_stem='05_forward_folded_uncertainty_transfer',
    y_title='Reco entries', ratio_nominal='Response statistical errors',
    ratio_range=(0.98, 1.02))


## 6. Common unsmeared starting point for selected toys

Toys 1, 3, and 5 (configurable) all begin from the same forward-folded expectation with copied experimental errors. Their curves intentionally coincide here. These exact one-based toy labels are used in every later plot.


In [ ]:
# Cell role: make the common pre-smearing state explicit for the selected toy identities.
selected_unsmeared_by_pt = {}
for toy_number in SELECTED_TOY_NUMBERS:
    selected_unsmeared_by_pt[toy_number] = [
        detached_clone(expectation_by_pt[i], f'hTutorialUnsmearedToy{toy_number}Eta{i}')
        for i in range(len(PTAVE_INTERVALS))]
step6_canvases, step6_objects = draw_per_pt(
    lambda i: {
        f'Toy {number} before smearing': selected_unsmeared_by_pt[number][i]
        for number in SELECTED_TOY_NUMBERS
    }, output_stem='06_selected_toys_before_smearing',
    y_title='Reco entries',
    ratio_nominal=f'Toy {SELECTED_TOY_NUMBERS[0]} before smearing',
    ratio_range=(0.98, 1.02))


## 7. The same selected toys after Gaussian smearing

Only bin contents are fluctuated. Every pseudo-experiment retains the copied experimental bin errors from step 5, making it possible to see central-value changes without changing the assigned uncertainty.


In [ ]:
# Cell role: generate one reproducible ensemble and plot the selected smeared toys.
toys = make_gaussian_toys(
    toy_expectation, n_toys=N_TOYS, random_seed=RANDOM_SEED, layout=layout,
    stop_on_negative=False, resample_negative=RESAMPLE_NEGATIVE_TOY_BIN,
    smear_zero_content=False)
selected_toy_eta = {
    number: [
        eta_block(toys[number - 1], training_reco_by_pt[i], i,
                  f'hTutorialSmearedToy{number}Eta{i}')
        for i in range(len(PTAVE_INTERVALS))]
    for number in SELECTED_TOY_NUMBERS
}
step7_canvases, step7_objects = draw_per_pt(
    lambda i: {
        'Forward-folded expectation': expectation_by_pt[i],
        **{f'Toy {number} after smearing': selected_toy_eta[number][i]
           for number in SELECTED_TOY_NUMBERS},
    }, output_stem='07_selected_toys_after_smearing',
    y_title='Reco entries', ratio_nominal='Forward-folded expectation')


## 8. Selected unfolded toys compared with toy truth

For every selected pseudo-experiment and configured iteration number, the purity-corrected toy is unfolded with the matched response and then divided by efficiency. Each pTave plot uses the same toy identities chosen in step 6.


In [ ]:
# Cell role: unfold the selected toys for all scan iterations and compare requested iterations to target truth.
signal_toys = [
    apply_purity_correction(toy, corrections.purity, name=f'hTutorialSignalToy{i + 1}')
    for i, toy in enumerate(toys)]
unfolded_selected = {number: {} for number in SELECTED_TOY_NUMBERS}
for toy_number in SELECTED_TOY_NUMBERS:
    toy_index = toy_number - 1
    for iterations in range(1, MAX_ITERATIONS + 1):
        matched = unfold_bayes(
            RooUnfold, matched_response, signal_toys[toy_index],
            iterations=iterations, handle_fakes=False,
            calculate_covariance=False, suppress_native_output=True,
            name=f'hTutorialToy{toy_number}UnfoldedMatchedIter{iterations}')
        inclusive = detached_clone(
            matched.histogram, f'hTutorialToy{toy_number}UnfoldedIter{iterations}')
        for global_bin in range(1, layout.n_global_bins + 1):
            efficiency = corrections.efficiency.GetBinContent(global_bin)
            if not math.isfinite(efficiency):
                raise ValueError(
                    f'Non-finite efficiency in global bin {global_bin}: {efficiency:g}')
            if efficiency <= 0.0:
                if (inclusive.GetBinContent(global_bin) != 0.0
                        or inclusive.GetBinError(global_bin) != 0.0):
                    pt_index, eta_bin = layout.indices(global_bin)
                    raise ValueError(
                        f'Cannot efficiency-correct populated global bin {global_bin} '
                        f'(pT block {pt_index}, eta bin {eta_bin}): '
                        f'efficiency={efficiency:g}')
                continue
            inclusive.SetBinContent(
                global_bin, inclusive.GetBinContent(global_bin) / efficiency)
            inclusive.SetBinError(
                global_bin, inclusive.GetBinError(global_bin) / efficiency)
        unfolded_selected[toy_number][iterations] = inclusive

step8_canvases = []
step8_objects = []
for toy_number in SELECTED_TOY_NUMBERS:
    canvases, objects = draw_per_pt(
        lambda i, toy_number=toy_number: {
            'Toy truth': scaled_truth_by_pt[i],
            **{
                f'Toy {toy_number}, {n} iteration' if n == 1
                else f'Toy {toy_number}, {n} iterations':
                eta_block(unfolded_selected[toy_number][n], training_truth_by_pt[i], i,
                          f'hTutorialToy{toy_number}UnfoldedEtaIter{n}Pt{i}')
                for n in DISPLAY_ITERATIONS
            },
        }, output_stem=f'08_toy_{toy_number}_iterations_vs_toy_truth',
        y_title='Truth entries', ratio_nominal='Toy truth', ratio_range=(0.5, 1.5))
    step8_canvases.extend(canvases)
    step8_objects.extend(objects)


## 9. Selected unfolded toys compared with original response-training Gen

This repeats step 8 against the original Gen distribution used to train the response. The response-training Gen is placed on the same global exposure scale as the toy truth solely for this yield-level comparison; its shape and binning are unchanged.


In [ ]:
# Cell role: compare the same unfolded toy/iteration set with the scaled original Gen prior.
scaled_training_truth = detached_clone(training_truth, 'hTutorialScaledTrainingTruth')
training_integral = scaled_training_truth.Integral()
toy_integral = toy_truth.Integral()
if training_integral <= 0.0 or toy_integral <= 0.0:
    raise ValueError('Truth integrals must be positive for the Gen comparison')
scaled_training_truth.Scale(toy_integral / training_integral)
scaled_training_truth_by_pt = [
    eta_block(scaled_training_truth, training_truth_by_pt[i], i,
              f'hTutorialScaledTrainingTruthEta{i}')
    for i in range(len(PTAVE_INTERVALS))]

step9_canvases = []
step9_objects = []
for toy_number in SELECTED_TOY_NUMBERS:
    canvases, objects = draw_per_pt(
        lambda i, toy_number=toy_number: {
            'Response-training Gen': scaled_training_truth_by_pt[i],
            **{
                f'Toy {toy_number}, {n} iteration' if n == 1
                else f'Toy {toy_number}, {n} iterations':
                eta_block(unfolded_selected[toy_number][n], training_truth_by_pt[i], i,
                          f'hTutorialToy{toy_number}VsGenEtaIter{n}Pt{i}')
                for n in DISPLAY_ITERATIONS
            },
        }, output_stem=f'09_toy_{toy_number}_iterations_vs_original_gen',
        y_title='Truth entries', ratio_nominal='Response-training Gen',
        ratio_range=(0.5, 1.5),
        curve_styles={
            'Response-training Gen': (ROOT.kBlue, 21),
            **{(f'Toy {toy_number}, {n} iteration' if n == 1
                else f'Toy {toy_number}, {n} iterations'):
               (ROOT.kBlack, (22, 26, 32)[index % 3])
               for index, n in enumerate(DISPLAY_ITERATIONS)},
        })
    step9_canvases.extend(canvases)
    step9_objects.extend(objects)


## 10. MSE, variance, and bias squared versus iteration

The complete common toy ensemble is unfolded at each iteration. The absolute plot averages the per-bin quantities in yield-squared units; the relative plot averages after dividing valid bins by truth squared. The first MSE minimum is selected.


In [ ]:
# Cell role: run the sequential ensemble scan and plot bias^2, variance, and MSE.
def progress(iterations, maximum):
    print(f'Sequential RooUnfold iteration {iterations}/{maximum}')

scan = scan_bayes_regularization(
    RooUnfold, matched_response, signal_toys, toy_truth,
    max_iterations=MAX_ITERATIONS, selection_mode='both',
    handle_fakes=False, calculate_covariance=False,
    efficiency=corrections.efficiency, progress=progress)
print(f'Absolute-MSE minimum: {scan.absolute_selected_iterations}')
print(f'Relative-MSE minimum: {scan.relative_selected_iterations}')
absolute_canvas, absolute_objects = draw_regularization_metrics(
    scan, mode='absolute', output=OUTPUT_DIR / '10_absolute_mse_variance_bias2.pdf',
    log_y=True)
relative_canvas, relative_objects = draw_regularization_metrics(
    scan, mode='relative', output=OUTPUT_DIR / '10_relative_mse_variance_bias2.pdf',
    log_y=True)
display(absolute_canvas); display(relative_canvas)


## 11. Optimal-iteration unfolded toys compared with toy truth

The absolute-MSE optimum is the primary iteration when `selection_mode='both'`. All selected toys are overlaid with the known target truth in every pTave interval.


In [ ]:
# Cell role: compare the same selected toys at the global optimum with target truth.
optimal_iterations = scan.selected_iterations
step11_canvases, step11_objects = draw_per_pt(
    lambda i: {
        'Toy truth': scaled_truth_by_pt[i],
        **{
            f'Unfolded toy {number} ({optimal_iterations} iter.)':
            eta_block(unfolded_selected[number][optimal_iterations], training_truth_by_pt[i], i,
                      f'hTutorialOptimalToy{number}VsTruthEta{i}')
            for number in SELECTED_TOY_NUMBERS
        },
    }, output_stem=f'11_optimal_{optimal_iterations}_unfolded_vs_toy_truth',
    y_title='Truth entries', ratio_nominal='Toy truth', ratio_range=(0.5, 1.5))


## 12. Optimal-iteration unfolded toys compared with original Gen

The same optimal iteration and the same selected toys are compared with the exposure-scaled response-training Gen distribution. This final view makes the regularization pull toward the response prior directly visible.


In [ ]:
# Cell role: compare the selected optimal-iteration results with original Gen.
step12_canvases, step12_objects = draw_per_pt(
    lambda i: {
        'Response-training Gen': scaled_training_truth_by_pt[i],
        **{
            f'Unfolded toy {number} ({optimal_iterations} iter.)':
            eta_block(unfolded_selected[number][optimal_iterations], training_truth_by_pt[i], i,
                      f'hTutorialOptimalToy{number}VsGenEta{i}')
            for number in SELECTED_TOY_NUMBERS
        },
    }, output_stem=f'12_optimal_{optimal_iterations}_unfolded_vs_original_gen',
    y_title='Truth entries', ratio_nominal='Response-training Gen',
    ratio_range=(0.5, 1.5),
    curve_styles={
        'Response-training Gen': (ROOT.kBlue, 21),
        **{f'Unfolded toy {number} ({optimal_iterations} iter.)':
           (ROOT.kBlack, (22, 26, 32)[index % 3])
           for index, number in enumerate(SELECTED_TOY_NUMBERS)},
    })

print(f'All tutorial figures written below {OUTPUT_DIR}')


## Interpretation checklist

- Confirm that the response matrix has the expected block structure and that Gen/Reco, purity, and efficiency are sensible in every pTave interval.
- Use steps 2 and 3 to separate the independent toy-truth shape from the response-training Bayesian prior.
- Verify in step 5 that copying experimental errors changes error bars, not central values.
- Verify in steps 6 and 7 that the selected toy identities are fixed and that smearing changes their central values while retaining assigned bin errors.
- Use steps 8 and 9 to see iteration-by-iteration convergence toward the target and the response prior.
- Treat an MSE minimum at 1 or `MAX_ITERATIONS` as a scan-boundary result; extend the scan without changing seed, toys, response, truth, or binning.
- Use steps 11 and 12 to compare the globally selected iteration with both truth roles in every pTave interval.
